In [6]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.weightstats import ztest

import scipy.stats as stats

In [7]:
df=pd.read_csv(r"C:\Users\shaiv\Downloads\data analytics projects\bayesian_ab_testing_platform\data\experiments\ab_experiment_dataset.csv")

In [8]:
#variant dataframes
variant_a = df[df["variant"]=="A"]
variant_b = df[df["variant"]=="B"]

Conversion Hypothesis Test

In [9]:
conversions = np.array([
    variant_a["converted"].sum(),
    variant_b["converted"].sum()
])

sample_sizes = np.array([
    len(variant_a),
    len(variant_b)
])

In [14]:
z_stat, p_value = proportions_ztest(
    conversions,sample_sizes
)

In [16]:
print("Conversion z-stat:",z_stat)
print("Conversion p-value:",p_value)

Conversion z-stat: -17.46826922814005
Conversion p-value: 2.4996313358356406e-68


In [20]:
alpha=0.05

In [22]:
if p_value<alpha:
    print("Result: Statistically significant")
else:
    print("Result: Not statistically significant")

Result: Statistically significant


In [24]:
#conversion uplift
conversion_a = variant_a["converted"].mean()
conversion_b = variant_b["converted"].mean()
uplift=conversion_b-conversion_a
print("Absolute Uplift:",uplift)

relative_uplift=(uplift/conversion_a)*100
print("Relative Uplift(%):",relative_uplift)

Absolute Uplift: 0.03191730375815334
Relative Uplift(%): 48.25315387875401


CTR Hypothesis Test

In [30]:
clicks = np.array([
    variant_a["clicked"].sum(),
    variant_b["clicked"].sum()
])

click_sample_sizes = np.array([
    len(variant_a),
    len(variant_b)
])

In [32]:
ctr_z_stat,ctr_p_value=proportions_ztest(
    clicks,
    click_sample_sizes
)

In [34]:
print("CTR Z-Statistic:", ctr_z_stat)
print("CTR P-Value:", ctr_p_value)

CTR Z-Statistic: -11.751397816347765
CTR P-Value: 6.946023740984026e-32


Revenue Hypothesis Test

In [54]:
df["experiment_revenue"].isnull().sum()

1

In [56]:
revenue_a = variant_a[
    "experiment_revenue"
].dropna()

revenue_b = variant_b[
    "experiment_revenue"
].dropna()

In [76]:
revenue_t_stat, revenue_p_value = stats.ttest_ind(
    revenue_b,
    revenue_a,
    equal_var=False
)

In [78]:
p_pool = (
    conversions.sum()/sample_sizes.sum()
)
standard_error = np.sqrt(
    p_pool*(1-p_pool)*
    (
        1/sample_sizes[0]+1/sample_sizes[1]
    )
)

In [80]:
margin_error = 1.96*standard_error
lower_bound=uplift-margin_error
upper_bound=uplift+margin_error

print("95% Confidence Interval:")
print(lower_bound,upper_bound)

95% Confidence Interval:
0.028336072295038234 0.035498535221268446


In [82]:
results = pd.DataFrame({
    "Metric": [
        "Conversion",
        "CTR",
        "Revenue"
    ],
    "P-Value": [
        p_value,
        ctr_p_value,
        revenue_p_value
    ]
})

results

,Metric,P-Value
0,Conversion,2.499631e-68
1,CTR,6.946024e-32
2,Revenue,4.895369e-07


## Frequentist Hypothesis Testing Conclusions

A classical frequentist A/B testing framework was applied to evaluate whether Variant B significantly outperformed Variant A across key business metrics including conversion rate, click-through rate (CTR), and revenue.

### Conversion Rate Testing

A proportion z-test was conducted to compare conversion probabilities between the two variants.

* Variant A Conversion Rate ≈ 6.94%
* Variant B Conversion Rate ≈ 9.82%

The resulting p-value was:

* p-value ≈ 2.50 × 10⁻⁶⁸

This p-value is far below the standard significance threshold of 0.05, indicating extremely strong statistical evidence against the null hypothesis.

Therefore, the null hypothesis of equal conversion probabilities was rejected, suggesting that Variant B produced a statistically significant improvement in conversions.

The observed uplift was approximately:

* Absolute uplift ≈ 3.19 percentage points
* Relative uplift ≈ 48%

This indicates that the treatment effect was not only statistically significant but also practically meaningful from a business perspective.

---

### CTR (Click Through Rate) Testing

A proportion z-test was also conducted for click-through behavior.

* Variant A CTR ≈ 11.98%
* Variant B CTR ≈ 14.85%

The resulting p-value was:

* p-value ≈ 6.95 × 10⁻³²

This extremely small p-value suggests that the observed CTR uplift was highly unlikely to occur due to random variation alone.

The results indicate that Variant B significantly improved customer engagement and interaction behavior.

---

### Revenue Testing

Since revenue is a continuous numerical metric, Welch’s t-test was used instead of a proportion z-test.

* Variant A Revenue Per User ≈ 160.97
* Variant B Revenue Per User ≈ 169.29

The resulting p-value was:

* p-value ≈ 4.90 × 10⁻⁷

This result indicates statistically significant revenue uplift under Variant B.

Although the revenue increase was smaller relative to the conversion uplift, the effect remained meaningful and consistent with realistic ecommerce experimentation outcomes.

---

## Overall Experiment Conclusion

Across all evaluated KPIs:

* conversion rate
* CTR
* revenue per user

Variant B consistently outperformed Variant A.

All hypothesis tests produced p-values significantly below the 0.05 significance threshold, indicating that the observed improvements were highly unlikely to be caused by random chance alone.

The experiment therefore provides strong statistical evidence that the Variant B treatment generated meaningful improvements in:

* customer engagement
* purchasing behavior
* monetization performance

Overall, the experimentation pipeline successfully simulated a realistic production-grade A/B testing workflow combining:

* behavioral uplift modeling
* statistical hypothesis testing
* business KPI evaluation
* experimentation analytics
